# YomiToku-Pro Table Semantic Parser in SageMaker

申込書・請求書・報告書といった帳票から、**表の意味構造**を抽出する解析エンジンです。枠線の有無によらず表のセルを直接検出し、次の情報を返します。

- `tables[].cells`: 検出したセル（`cell_id` をキーにした辞書）。テキスト・役割(header / cell / empty / group)・行列位置を持ちます
- `tables[].kv_items`: 「項目名 → 値」の組。キー・値は**セルID**で参照します
- `tables[].grids`: 行・列のグリッド。列ヘッダーと各セルの配置を**セルID**で持ちます
- `paragraphs`: 表の外にある段落
- `words`: 単語単位のOCR結果

AWS Marketplaceでサブスク可能であり、AWS SageMaker上でWeb API形式で利用できます。

本ノートブックでは以下の内容を解説します。
- AWS SageMaker上にYomiToku-Pro Table Semantic ParserのEndpointをデプロイする方法
- AWSの認証の設定
- YomiToku-Clientからエンドポイントを呼び出し、レスポンスを取得・保存する方法

> **現在の対応範囲**: YomiToku-Clientは、エンドポイントの呼び出しとレスポンスの取得・JSON保存に対応しています。セルIDを解決した構造化データの取得、CSV / Markdown / HTML への変換、可視化は今後の対応です。

## 事前準備
### YomiToku-Clientのインストール

In [ ]:
!pip install yomitoku-client

### AWSの認証の設定
[こちら](https://mlism-inc.github.io/yomitoku-client/iam-doc/)のドキュメントを参照して、AWSの認証の設定を行ってください

### SageMakerエンドポイントの作成
事前に **CloudFormation** または **SageMaker** コンソールを使って、このモデルのエンドポイントを作成します。

[こちら](https://mlism-inc.github.io/yomitoku-client/deploy-yomitoku-pro/)のドキュメントを参照し、AWS SageMakerのエンドポイントを作成してください。

Table Semantic Parserは文書解析(Document Analyzer)とは**別のモデルパッケージ**です。エンドポイントの作成手順は共通で、サブスクライブしたTable Semantic ParserのModel Package ARNを指定します。
文書解析のエンドポイントと同時に立ち上げて共存させることもできます。

## エンドポイントの呼び出し

`YomitokuClient`は文書解析APIとTable Semantic Parserで共通です。エンドポイント名にTable Semantic Parserのエンドポイントを指定します。

#### 引数
- `endpoint`: SageMakerのエンドポイント名を指定します。
- `region`: エンドポイントを立ち上げたAWS REGIONを指定します。

In [ ]:
from yomitoku_client import YomitokuClient

ENDPOINT_NAME = "my-tsp-endpoint"
AWS_REGION = "ap-northeast-1"

target_file = "sample/table.jpg"

with YomitokuClient(
    endpoint=ENDPOINT_NAME,
    region=AWS_REGION,
) as client:
    # 非同期呼び出し
    result = await client.analyze_async(
        target_file,
        # page_index=[0, 1, 2], # ページ指定オプション (省略時は全ページ解析)
    )

### データ変換
エンドポイントからは生のJSONデータ(辞書形式)が取得されます。
`parse_table_semantic_parser()`を使用して、スキーマ検証を行いながらpydantic modelのインスタンスに変換します。


In [ ]:
from yomitoku_client import parse_table_semantic_parser

model = parse_table_semantic_parser(result)

page = model.pages[0]
print(f"ページ数: {len(model.pages)}")
print(f"検出したテーブル数: {len(page.tables)}")
print(f"段落数: {len(page.paragraphs)} / 単語数: {len(page.words)}")

## レスポンスの構造

### テーブルとセル

`cells` は `cell_id` をキーにした辞書です。`kv_items` と `grids` はテキストではなく**セルID**を参照するため、テキストは `cells[cell_id].contents` から引きます。

In [ ]:
table = page.tables[0]

print(f"table id={table.id} style={table.style} box={table.box}")
print(f"cells={len(table.cells)} kv_items={len(table.kv_items)} grids={len(table.grids)}")

for cell_id, cell in list(table.cells.items())[:5]:
    print(cell_id, cell.role, cell.box, repr(cell.contents))

### kv_items（キーと値）

`key` はキーセルのID配列（外側の親ヘッダーから順）、`value` は値セルのIDです。

In [ ]:
for kv_item in table.kv_items[:5]:
    key_ids = [kv_item.key] if isinstance(kv_item.key, str) else kv_item.key
    key_text = " / ".join(table.cells[k].contents or "" for k in key_ids if k in table.cells)
    value_text = table.cells[kv_item.value].contents if kv_item.value in table.cells else None
    print(f"{key_text} -> {value_text}")

### grids（行と列）

`col_headers` は列ごとの列ヘッダーセルID、`data` は行×列に並べたセルIDです。

In [ ]:
for grid in table.grids:
    print(f"--- grid {grid.id}: {grid.n_row} rows x {grid.n_col} cols ---")

    headers = [
        " ".join(table.cells[h].contents or "" for h in header if h in table.cells)
        for header in grid.col_headers
    ]
    print(headers)

    for row in grid.data:
        print([
            (table.cells[cell_id].contents if cell_id in table.cells else None)
            for cell_id in row
        ])

### 段落と単語

表の外にあるテキストは `paragraphs`、単語単位のOCR結果は `words` で取得できます。

In [ ]:
for paragraph in page.paragraphs[:5]:
    print(paragraph.box, repr(paragraph.contents))

for word in page.words[:5]:
    print(word.points[0], repr(word.content), word.direction, round(word.rec_score, 3))

## JSON として保存

APIのレスポンスをそのままJSONとして保存します。

**引数**
- `output_path`: 出力ファイルのパスを指定します。パスにフォルダが含まれる場合は指定フォルダに出力します。
- `mode`: 出力モード(`combine` or `separate`)を指定します。default: `combine`
    - `combine`モードは複数ページの読み取り結果を単一ファイルにまとめて出力
    - `separate`モードは複数ページをそれぞれのページに分割して出力
- `page_index`: 出力対象のページを指定します。(e.g. [0,1,2])

In [ ]:
# 読み取り結果をjson形式で保存
out = model.to_json(output_path="table.json")

# demoフォルダ内に保存
# out = model.to_json(output_path="demo/table.json")

# ページごとに分割して保存（mode="separate"）
# out = model.to_json(output_path="table.json", mode="separate")

# 指定のページのみ保存
# out = model.to_json(output_path="table.json", page_index=[0, 2])

## バッチ処理の実行

`analyze_batch_async`は文書解析APIと共通です。フォルダ内のファイルを並列に解析し、生のJSONと処理ログ(`process_log.jsonl`)を出力します。

### 機能の特長
- **フォルダ単位での一括解析** : 指定ディレクトリ内のPDF・画像ファイルを自動で検出し、並列処理を実行。
- **中間ログ出力（process_log.jsonl）**: 各ファイルの処理結果・成功可否・エラー内容を1行ごとに記録。
- **上書き制御**: 既に解析済みのファイルはスキップ（overwrite=False）設定で効率化。
- **再実行対応**: ログをもとに、失敗したファイルのみを再解析する運用が容易。

In [ ]:
import json
import os

from yomitoku_client import YomitokuClient, parse_table_semantic_parser

# 入出力設定
target_dir = "sample"
outdir = "output"

# SageMakerエンドポイント設定
ENDPOINT_NAME = "my-tsp-endpoint"
AWS_REGION = "ap-northeast-1"

async with YomitokuClient(
    endpoint=ENDPOINT_NAME,
    region=AWS_REGION,
) as client:
    # 1. `sample`フォルダ内の画像を対象としてバッチ処理を実行
    # 2. 中間出力の辞書データを`outdir`に格納
    # 3. 処理ログを`output/process_log.jsonl`に記録
    await client.analyze_batch_async(
        input_dir=target_dir,
        output_dir=outdir,
    )


with open(os.path.join(outdir, "process_log.jsonl"), encoding="utf-8") as f:
    logs = [json.loads(line) for line in f if line.strip()]

out_formatted = os.path.join(outdir, "formatted")
os.makedirs(out_formatted, exist_ok=True)

# 4. ログを読んで処理に成功したファイルのみスキーマ検証してJSONに保存
for log in logs:
    if not log.get("success"):
        continue

    with open(log["output_path"], encoding="utf-8") as rf:
        raw_result = json.load(rf)

    doc = parse_table_semantic_parser(raw_result)

    base = os.path.splitext(os.path.basename(log["file_path"]))[0]
    doc.to_json(output_path=os.path.join(out_formatted, f"{base}.json"))

## CLIから利用する

CLIでもTable Semantic Parserを利用できます。`--api table-semantic-parser`を指定してください(省略時は文書解析API)。
出力はAPIレスポンスそのままのJSONです。

```bash
# 単一ファイルの解析
yomitoku-client single sample/table.jpg \
    --api table-semantic-parser \
    --endpoint my-tsp-endpoint \
    --region ap-northeast-1 \
    --file_format json \
    --output_dir output

# フォルダ単位のバッチ解析
yomitoku-client batch \
    --api table-semantic-parser \
    --input_dir sample \
    --output_dir output \
    --endpoint my-tsp-endpoint \
    --file_format json
```